In [ ]:
!pip install chromadb sentence-transformers pypdf scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

import chromadb

from pypdf import PdfReader

In [ ]:
df = pd.read_csv("fertilizer_recommendation_dataset.csv")

df.head()

,Temperature,Moisture,Rainfall,PH,Nitrogen,Phosphorous,Potassium,Carbon,Soil,Crop,Fertilizer,Remark
0,50.179845,0.725893,205.600816,6.227358,66.701872,76.963560,96.429065,0.496300,Loamy Soil,rice,Compost,Enhances organic matter and improves soil stru...
1,21.633318,0.721958,306.081601,7.173131,71.583316,163.057636,148.128347,1.234242,Loamy Soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
2,23.060964,0.685751,259.336414,7.380793,75.709830,62.091508,80.308971,1.795650,Peaty Soil,rice,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...
3,26.241975,0.755095,212.703513,6.883367,78.033687,151.012521,153.005712,1.517556,Loamy Soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
4,21.490157,0.730672,268.786767,7.578760,71.765123,66.257371,97.000886,1.782985,Peaty Soil,rice,Organic Fertilizer,"Enhances fertility naturally, ideal for peaty ..."


In [ ]:
print(df.columns)

Index(['Temperature', 'Moisture', 'Rainfall', 'PH', 'Nitrogen', 'Phosphorous',
       'Potassium', 'Carbon', 'Soil', 'Crop', 'Fertilizer', 'Remark'],
      dtype='object')


In [ ]:
df = df.dropna()

df["Crop"] = df["Crop"].str.lower()
df["Soil"] = df["Soil"].str.lower()

df.head()

,Temperature,Moisture,Rainfall,PH,Nitrogen,Phosphorous,Potassium,Carbon,Soil,Crop,Fertilizer,Remark
0,50.179845,0.725893,205.600816,6.227358,66.701872,76.963560,96.429065,0.496300,loamy soil,rice,Compost,Enhances organic matter and improves soil stru...
1,21.633318,0.721958,306.081601,7.173131,71.583316,163.057636,148.128347,1.234242,loamy soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
2,23.060964,0.685751,259.336414,7.380793,75.709830,62.091508,80.308971,1.795650,peaty soil,rice,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...
3,26.241975,0.755095,212.703513,6.883367,78.033687,151.012521,153.005712,1.517556,loamy soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
4,21.490157,0.730672,268.786767,7.578760,71.765123,66.257371,97.000886,1.782985,peaty soil,rice,Organic Fertilizer,"Enhances fertility naturally, ideal for peaty ..."


In [ ]:
features = [
    "Nitrogen",
    "Phosphorous",
    "Potassium",
    "PH"
]

In [ ]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    df[features]
)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_fertilizer(
        crop,
        soil,
        n,
        p,
        k,
        ph,
        top_k=10):

    crop = crop.lower()
    soil = soil.lower()

    crop_df = df[
        (df["Crop"] == crop)
    ]

    if len(crop_df) == 0:
        return None

    crop_scaled = scaler.transform(
        crop_df[features]
    )

    user_input = pd.DataFrame(
        [[n, p, k, ph]],
        columns=features
    )

    user_vector = scaler.transform(
        user_input
    )

    similarities = cosine_similarity(
        user_vector,
        crop_scaled
    )[0]

    crop_df = crop_df.copy()

    crop_df["Similarity"] = similarities

    result = crop_df.sort_values(
        by="Similarity",
        ascending=False
    )

    return result.head(top_k)

In [ ]:
recommend_fertilizer(
    crop="rice",
    soil="clayey",
    n=25,
    p=20,
    k=15,
    ph=6.5
)

,Temperature,Moisture,Rainfall,PH,Nitrogen,Phosphorous,Potassium,Carbon,Soil,Crop,Fertilizer,Remark,Similarity
80,27.858491,0.820361,201.020526,3.908613,35.527718,39.256980,50.187523,2.283017,peaty soil,rice,Urea,"Provides high nitrogen, ideal for rapid leafy ...",0.686310
39,23.391164,0.808853,306.081601,3.908613,35.527718,47.956150,66.919503,1.560150,peaty soil,rice,Urea,"Provides high nitrogen, ideal for rapid leafy ...",0.634520
96,23.953411,0.769098,280.559953,6.611215,35.527718,74.151730,114.207173,1.589816,loamy soil,rice,Urea,"Provides high nitrogen, ideal for rapid leafy ...",0.603739
52,23.031840,0.901894,175.775862,7.379071,35.527718,91.649919,103.834546,1.368341,loamy soil,rice,Urea,"Provides high nitrogen, ideal for rapid leafy ...",0.586931
14,50.179845,0.838430,208.650127,6.877079,35.527718,144.007817,149.258134,1.484953,loamy soil,rice,Urea,"Provides high nitrogen, ideal for rapid leafy ...",0.224225
66,22.061359,0.926694,233.565156,5.569052,68.446876,-37.649739,54.323607,2.041603,peaty soil,rice,DAP,"Rich in phosphorus, essential for root develop...",0.091878
58,21.863397,0.713272,186.446773,7.507470,67.752258,141.919592,-20.509108,1.702499,loamy soil,rice,Muriate of Potash,"High potassium content, improves fruit and flo...",-0.012146
83,25.532500,0.972361,194.776800,6.021260,72.775054,-37.649739,68.327916,1.185096,peaty soil,rice,DAP,"Rich in phosphorus, essential for root develop...",-0.089907
49,23.307913,0.972361,250.253012,7.502742,80.809203,68.071117,-20.509108,1.194465,peaty soil,rice,Muriate of Potash,"High potassium content, improves fruit and flo...",-0.193771
19,23.934519,0.750684,283.192870,5.592620,80.334366,63.621699,-20.509108,0.744585,acidic soil,rice,Muriate of Potash,"High potassium content, improves fruit and flo...",-0.198435


In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 65.2 MB/s eta 0:00:00


In [ ]:
import fitz

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text

In [ ]:
icar1 = extract_pdf_text("/content/ICAR_Rabi.pdf")

icar2 = extract_pdf_text("/content/ICAR_Nutrient.pdf")

full_text = icar1 + "\n" + icar2

print(len(full_text))

203051


In [ ]:
print(full_text[:1000])


ix
Message
iii
Foreword
v
Preface
vii
SOIL AND NUTRIENT MANAGEMENT
On-line soil fertility maps of different states and fertilizer
1
recommendation system for targeted yields of crops
PUSA soil-test fertilizer recommendation (STFR) Meter
2
Iron (Fe) enrichment in rice, maize and pulses
3
Techniques for correcting zinc (Zn) deficiency
4
Balanced fertilization through by sulphur application
5
Ameliorating deficiency of Manganese (Mn) in field crops
7
Bio-enriched compost
8
Technology for preparation of enriched compost
9
Phosphate solubilizers (Trichoderma sp. and Penicillium sp.)
10
Vermicomposting technology for recycling of organic wastes
11
Microbial based bio-nutrient package for rice
12
Compost inoculant
14
Liquid inoculants technology of biofertilizer organisms
15
Bacterial inoculant for quality seedling production of apple
16
Bacterial inoculant for rejuvenation of white root rot infested
17
apple orchard
Kit for assessing soil organic-carbon
18
Kit for assessing compostability o

In [ ]:
def chunk_text(
        text,
        chunk_size=500):

    chunks=[]

    for i in range(
        0,
        len(text),
        chunk_size
    ):

        chunks.append(
            text[i:i+chunk_size]
        )

    return chunks

In [ ]:
chunks = chunk_text(full_text)

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
client = chromadb.Client()

collection = client.create_collection(
    "icar_fertilizer1"
)

In [ ]:
embeddings = embedding_model.encode(
    chunks
)

collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=[
        str(i)
        for i in range(len(chunks))
    ]
)

In [ ]:
def search_icar(query):

    query_embedding = embedding_model.encode(
        query
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=3
    )

    return results["documents"][0]

In [ ]:
search_icar(
    "rice fertilizer management"
)

['wm@icar.org.in\nNatural Resource Management: Technologies Ready for Commercialization\n24\nTechnology of Direct Seeded Rice (DSR)\nSalient features\nSowing of rice seeds done directly into field\nthough multi-crop planters or zero till seed –cum-\nfertilizer drill in dry or optimum moisture\nconditions. If sown dry, it has to be followed by\nirrigation.\nSeed rate: 20-25 kg per ha\nNutrient management: Basal @200 kg NPK\n(12:32:16) should be drilled at sowing + 50 kg\nMOP ha-1. 25 kg zinc sulphate ha-1 to be',
 'ies in soils, site specific nutrient\nmanagement, which is considered as Fertilizer Best Management Practice, needs to be promoted\nto improve soil health and crop productivity. Inadequate and unreliable soil-testing facilities,\npoor awareness of farmers about balanced plant nutrition and lack of appropriate policy are\nthe major constraints in adoption of Fertilizer Best Management Practices. It is realized that\nthe soil-testing service has not made the desired impact and 

In [ ]:
def hybrid_recommendation(
        crop,
        soil,
        n,
        p,
        k,
        ph):

    ferts = recommend_fertilizer(
        crop,
        soil,
        n,
        p,
        k,
        ph
    )

    if ferts is None:
        return {
            "error": "Crop not found"
        }

    top_fertilizers = (
        ferts["Fertilizer"]
        .value_counts()
        .head(3)
        .index
        .tolist()
    )

    rag_query = f"""
    Crop: {crop}

    Soil Type: {soil}

    Nitrogen: {n}
    Phosphorous: {p}
    Potassium: {k}
    pH: {ph}

    Recommended Fertilizers:
    {' '.join(top_fertilizers)}

    Fertilizer dosage
    Nutrient management
    Application schedule
    Precautions
    """

    context = search_icar(
        rag_query
    )

    return {
        "recommended_fertilizers":
            top_fertilizers,

        "top_matches":
            ferts[
                [
                    "Fertilizer",
                    "Similarity",
                    "Remark"
                ]
            ].head(5),

        "icar_context":
            context
    }

In [ ]:
def show_recommendation(result):

    print("="*60)

    print("\nTOP RECOMMENDED FERTILIZERS\n")

    for i, fert in enumerate(
        result["recommended_fertilizers"],
        start=1
    ):
        print(f"{i}. {fert}")

    print("\n" + "="*60)

    print("\nTOP MATCHING DATASET RECORDS\n")

    print(
        result["top_matches"]
    )

    print("\n" + "="*60)

    print("\nICAR KNOWLEDGE RETRIEVED\n")

    for chunk in result["icar_context"]:
        print(chunk[:800])
        print("\n")

In [ ]:
result = hybrid_recommendation(
    crop="rice",
    soil="clayey",
    n=25,
    p=20,
    k=15,
    ph=6.5
)

show_recommendation(result)


TOP RECOMMENDED FERTILIZERS

1. Urea
2. Muriate of Potash
3. DAP


TOP MATCHING DATASET RECORDS

   Fertilizer  Similarity                                             Remark
80       Urea    0.686310  Provides high nitrogen, ideal for rapid leafy ...
39       Urea    0.634520  Provides high nitrogen, ideal for rapid leafy ...
96       Urea    0.603739  Provides high nitrogen, ideal for rapid leafy ...
52       Urea    0.586931  Provides high nitrogen, ideal for rapid leafy ...
14       Urea    0.224225  Provides high nitrogen, ideal for rapid leafy ...


ICAR KNOWLEDGE RETRIEVED

o
imbalance of nutrients and causing Zn
deficiency . Without phosphorus, the cost of
inputs will be reduced.
Performance results
Grain yield of rice was not influenced by
different doses of P where the soil available P
is high.
Natural Resource Management: Technologies Ready for Commercialization
26
Cost of technology
Amount saved without “P” application is R2,784/ha (Recommended dose is
80 kg P2O5/ha).
Impac

In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_OPENROUTER_API_KEY_HERE",
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
def generate_advisory(
    crop,
    soil,
    n,
    p,
    k,
    ph,
    fertilizers,
    context
):

    prompt = f"""
You are an agricultural expert.

Crop: {crop}
Soil: {soil}

N={n}
P={p}
K={k}
pH={ph}

Recommended Fertilizers:
{', '.join(fertilizers)}

ICAR Context:
{context}

Generate:

1. Soil Analysis
2. Recommended Fertilizers
3. Why they are recommended
4. Application schedule
5. Precautions

Keep it simple and practical.
"""

    response = client.chat.completions.create(
        model="meta-llama/llama-3.1-8b-instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
result = hybrid_recommendation(
    crop="rice",
    soil="clayey",
    n=25,
    p=20,
    k=15,
    ph=6.5
)

advisory = generate_advisory(
    crop="rice",
    soil="clayey",
    n=25,
    p=20,
    k=15,
    ph=6.5,
    fertilizers=result["recommended_fertilizers"],
    context="\n".join(result["icar_context"])
)

print(advisory)

Here's the information you requested:

**Soil Analysis**

* Soil Type: Clayey
* Soil pH: 6.5
* Available Nutrients:
	+ Nitrogen (N): 25
	+ Phosphorus (P): 20
	+ Potassium (K): 15

**Recommended Fertilizers**

1. Urea
2. Muriate of Potash (K)
3. DAP (Di-Ammonium Phosphate)

**Why they are recommended**

* Urea: To supplement nitrogen, which is essential for rice growth.
* Muriate of Potash (K): To supply potassium, which is necessary for rice growth and yield.
* DAP: Although the soil has sufficient P, DAP is recommended to maintain soil P levels and to prevent zinc deficiency.

**Application Schedule**

* Apply urea and muriate of potash at sowing time (15 kg N and 30 kg K per hectare).
* Apply DAP at tillering stage (50 kg per hectare).

**Precautions**

* Before applying fertilizers, it's essential to test the soil pH and adjust it if necessary.
* Apply fertilizers in accordance with the recommended application schedule to avoid excessive nutrient application.
* Monitor the soil's nu